# SMS Spam Classification with Classical NLP

## 1. Project Overview


In [16]:
import pandas as pd
import numpy as np

import nltk

## 2. Dataset and Exploratory Data Analysis


In [17]:
df = pd.read_csv(
    "../data/SMSSpamCollection",
    sep="\t",
    header=None
)

df.columns = ["label", "message"]
df.head()


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [18]:
df.size

11144

In [19]:
df.label.value_counts()

,count
label,
ham,4825
spam,747


In [20]:
df.shape

(5572, 2)

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   label    5572 non-null   object
 1   message  5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [22]:
df.isnull().sum()

,0
label,0
message,0


## 3. Text Preprocessing


In [23]:
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [24]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [25]:
from nltk.tokenize import word_tokenize

In [26]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [27]:
from nltk.corpus import stopwords

In [28]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

In [29]:
def preprocess_text(text):

  #Step 1:tokenize
  tokens = word_tokenize(text)

  #Step 2:lowercase
  lower_tokens = [
      token.lower()
      for token in tokens
  ]

  #Step 3:remove punctuation / non-alphabetic tokens
  clean_tokens = [
      token
      for token in lower_tokens
      if token.isalpha()
  ]

  #Step 4:remove stopwords
  filtered_tokens = [
      token
      for token in clean_tokens
      if token not in english_stopwords
  ]

  #Step 5: stemming
  stemmed_tokens = [
      stemmer.stem(filtered_tokens)
      for filtered_tokens in filtered_tokens
  ]

  return stemmed_tokens

In [30]:
import nltk
from nltk.corpus import stopwords

nltk.download("stopwords")

english_stopwords = set(stopwords.words("english"))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [31]:
df["processed_tokens"] = df["message"].apply(preprocess_text)

In [32]:
df[["message", "processed_tokens"]].sample(5)

,message,processed_tokens
2802,Depends on where u going lor.,"[depend, u, go, lor]"
4188,Dear got bus directly to calicut,"[dear, got, bu, directli, calicut]"
419,"Alright, I'll head out in a few minutes, text ...","[alright, head, minut, text, meet]"
2796,How will I creep on you now? ;_;,[creep]
4472,Wa... U so efficient... Gee... Thanx...,"[wa, u, effici, gee, thanx]"


## 4. Bag of Words Baseline


In [33]:
from sklearn.feature_extraction.text import CountVectorizer

In [34]:
df["processed_message"] = df["processed_tokens"].apply(
    lambda tokens: " ".join(tokens)
)

In [35]:
X = df["processed_message"]
y = df["label"]

In [36]:
from sklearn.model_selection import train_test_split

In [37]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [38]:
print(X_train.shape)
print(X_test.shape)

(4457,)
(1115,)


In [39]:
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

label
ham     0.865829
spam    0.134171
Name: proportion, dtype: float64
label
ham     0.866368
spam    0.133632
Name: proportion, dtype: float64


In [40]:
bow_vectorizer = CountVectorizer()

In [41]:
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

In [42]:
print(X_train_bow.shape)
print(X_test_bow.shape)

(4457, 5205)
(1115, 5205)


In [43]:
vocabulary = bow_vectorizer.get_feature_names_out()

print(len(vocabulary))
print(vocabulary[:50])

5205
['aa' 'aah' 'aaniy' 'aathi' 'ab' 'abbey' 'abdomen' 'abeg' 'abel'
 'aberdeen' 'abi' 'abil' 'abiola' 'abj' 'abl' 'abnorm' 'abouta' 'absenc'
 'absolut' 'absolutli' 'abstract' 'abt' 'abta' 'aburo' 'abus' 'ac'
 'academ' 'acc' 'accent' 'accentur' 'accept' 'access' 'accid' 'accident'
 'accommodationvouch' 'accomod' 'accordin' 'accordingli' 'account'
 'accumul' 'ach' 'achan' 'acknowledg' 'acnt' 'aco' 'across' 'act' 'actin'
 'action' 'activ']


In [44]:
from sklearn.linear_model import LogisticRegression

In [45]:
bow_model = LogisticRegression(
    max_iter=1000
)

In [46]:
bow_model.fit(
    X_train_bow,
    y_train
)

LogisticRegression(max_iter=1000)

In [47]:
y_pred_bow = bow_model.predict(X_test_bow)
print(y_pred_bow[:20])

['ham' 'ham' 'ham' 'spam' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham'
 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham']


In [48]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [49]:
bow_accuracy = accuracy_score(
    y_test,
    y_pred_bow
)

bow_precision = precision_score(
    y_test,
    y_pred_bow,
    pos_label="spam"
)

bow_recall = recall_score(
    y_test,
    y_pred_bow,
    pos_label="spam"
)

bow_f1 = f1_score(
    y_test,
    y_pred_bow,
    pos_label="spam"
)

In [50]:
print(bow_accuracy)
print(bow_precision)
print(bow_recall)
print(bow_f1)

0.9811659192825112
0.9776119402985075
0.8791946308724832
0.9257950530035336


In [51]:
cm_bow = confusion_matrix(
    y_test,
    y_pred_bow,
    labels=["ham", "spam"]
)

print(cm_bow)

[[963   3]
 [ 18 131]]


## 5. N-gram Experiments


In [52]:
ngram_vectorizer = CountVectorizer(
    ngram_range=(1, 2)
)

In [53]:
X_train_ngram = ngram_vectorizer.fit_transform(
    X_train
)

X_test_ngram = ngram_vectorizer.transform(
    X_test
)

In [54]:
print(X_train_bow.shape)
print(X_train_ngram.shape)

(4457, 5205)
(4457, 28244)


In [55]:
ngram_model = LogisticRegression(
    max_iter=1000
)

In [56]:
ngram_model.fit(X_train_ngram, y_train)

LogisticRegression(max_iter=1000)

In [57]:
y_pred_ngram = ngram_model.predict(X_test_ngram)

print(y_pred_ngram[:20])

['ham' 'ham' 'ham' 'spam' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham'
 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham']


In [58]:
ngram_accuracy = accuracy_score(
    y_test,
    y_pred_ngram
)

ngram_precision = precision_score(
    y_test,
    y_pred_ngram,
    pos_label="spam"
)

ngram_recall = recall_score(
    y_test,
    y_pred_ngram,
    pos_label="spam"
)

ngram_f1 = f1_score(
    y_test,
    y_pred_ngram,
    pos_label="spam"
)

In [59]:
print(ngram_accuracy)
print(ngram_precision)
print(ngram_recall)
print(ngram_f1)

0.9766816143497757
0.992
0.8322147651006712
0.9051094890510949


## 6. TF-IDF Experiments


In [60]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [61]:
tfidf_vectorizer = TfidfVectorizer()

In [62]:
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

In [63]:
print(X_train_tfidf.shape)
print(X_test_tfidf.shape)

(4457, 5205)
(1115, 5205)


In [64]:
tfidf_model = LogisticRegression(
    max_iter=1000
)

In [65]:
tfidf_model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

In [66]:
y_pred_tfidf = tfidf_model.predict(X_test_tfidf)

print(y_pred_tfidf[:20])

['ham' 'ham' 'ham' 'spam' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham'
 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham']


In [67]:
tfidf_accuracy = accuracy_score(
    y_test,
    y_pred_tfidf
)

tfidf_precision = precision_score(
    y_test,
    y_pred_tfidf,
    pos_label="spam"
)

tfidf_recall = recall_score(
    y_test,
    y_pred_tfidf,
    pos_label="spam"
)

tfidf_f1 = f1_score(
    y_test,
    y_pred_tfidf,
    pos_label="spam"
)

In [68]:
print(tfidf_accuracy)
print(tfidf_precision)
print(tfidf_recall)
print(tfidf_f1)

0.9614349775784753
1.0
0.7114093959731543
0.8313725490196079


## 7. TF-IDF + N-grams Experiments


In [69]:
tfidf_ngram_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2)
)

In [70]:
X_train_tfidf_ngram = tfidf_ngram_vectorizer.fit_transform(X_train)
X_test_tfidf_ngram = tfidf_ngram_vectorizer.transform(X_test)

In [71]:
print(X_train_tfidf_ngram.shape)
print(X_test_tfidf_ngram.shape)

(4457, 28244)
(1115, 28244)


In [72]:
tfidf_ngram_model = LogisticRegression(
    max_iter=1000
)

In [73]:
tfidf_ngram_model.fit(X_train_tfidf_ngram, y_train)

LogisticRegression(max_iter=1000)

In [74]:
y_pred_tfidf_ngram = tfidf_ngram_model.predict(X_test_tfidf_ngram)

print(y_pred_tfidf_ngram[:20])

['ham' 'ham' 'ham' 'spam' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham'
 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham']


In [75]:
tfidf_ngram_accuracy = accuracy_score(
    y_test,
    y_pred_tfidf_ngram
)

tfidf_ngram_precision = precision_score(
    y_test,
    y_pred_tfidf_ngram,
    pos_label="spam"
)

tfidf_ngram_recall = recall_score(
    y_test,
    y_pred_tfidf_ngram,
    pos_label="spam"
)

tfidf_ngram_f1 = f1_score(
    y_test,
    y_pred_tfidf_ngram,
    pos_label="spam"
)

In [76]:
print(tfidf_ngram_accuracy)
print(tfidf_ngram_precision)
print(tfidf_ngram_recall)
print(tfidf_ngram_f1)

0.9587443946188341
0.9904761904761905
0.697986577181208
0.8188976377952756


## 8. Error Analysis


In [77]:
print(X_test.index[:10])
print(y_test.index[:10])

Index([2825, 3695, 3904, 576, 2899, 3456, 5128, 919, 2505, 17], dtype='int64')
Index([2825, 3695, 3904, 576, 2899, 3456, 5128, 919, 2505, 17], dtype='int64')


In [78]:
original_test_messages = df.loc[
    X_test.index,
    "message"
]

In [79]:
results_df = pd.DataFrame({
    "message": original_test_messages.reset_index(drop=True),
    "processed_message": X_test.reset_index(drop=True),
    "actual": y_test.reset_index(drop=True),
    "predicted": y_pred_bow
})

In [80]:
errors = results_df[
    results_df["actual"] != results_df["predicted"]
]

print(errors.shape)

(21, 4)


In [81]:
print(errors.shape)

(21, 4)


In [82]:
false_positives = results_df[
    (results_df["actual"] == "ham") &
    (results_df["predicted"] == "spam")
]

print(false_positives.shape)

(3, 4)


In [83]:
false_negatives = results_df[
    (results_df["actual"] == "spam") &
    (results_df["predicted"] == "ham")
]

print(false_negatives.shape)

(18, 4)


In [84]:
print(false_positives.shape)
print(false_negatives.shape)

(3, 4)
(18, 4)


In [85]:
for message in false_positives["message"]:
    print(message)
    print("-----")

I'm always on yahoo messenger now. Just send the message to me and i.ll get it you may have to send it in the mobile mode sha but i.ll get it. And will reply.
-----
Let Ur Heart Be Ur Compass Ur Mind Ur Map Ur Soul Ur Guide And U Will Never loose in world....gnun - Sent via WAY2SMS.COM
-----
Yetunde, i'm sorry but moji and i seem too busy to be able to go shopping. Can you just please find some other way to get what you wanted us to get. Please forgive me. You can reply free via yahoo messenger.
-----


In [86]:
for message in false_negatives["message"]:
    print(message)
    print("-----")

FreeMsg Hey there darling it's been 3 week's now and no word back! I'd like some fun you up for it still? Tb ok! XxX std chgs to send, £1.50 to rcv
-----
it to 80488. Your 500 free text messages are valid until 31 December 2005.
-----
ringtoneking 84484
-----
Sorry I missed your call let's talk when you have the time. I'm on 07090201529
-----
Latest News! Police station toilet stolen, cops have nothing to go on!
-----
Can U get 2 phone NOW? I wanna chat 2 set up meet Call me NOW on 09096102316 U can cum here 2moro Luv JANE xx Calls£1/minmoremobsEMSPOBox45PO139WA
-----
More people are dogging in your area now. Call 09090204448 and join like minded guys. Why not arrange 1 yourself. There's 1 this evening. A£1.50 minAPN LS278BB
-----
For sale - arsenal dartboard. Good condition but no doubles or trebles!
-----
Email AlertFrom: Jeri StewartSize: 2KBSubject: Low-cost prescripiton drvgsTo listen to email call 123
-----
FreeMsg>FAV XMAS TONES!Reply REAL
-----
1000's of girls many local 2 u wh

## 9. Numeric Feature Engineering


In [87]:
df["has_digit"] = df["message"].apply(
    lambda message: any(char.isdigit() for char in message)
)

In [88]:
print(df["has_digit"].value_counts())

has_digit
False    4109
True     1463
Name: count, dtype: int64


In [89]:
df.groupby("label")["has_digit"].mean()

,has_digit
label,
ham,0.156477
spam,0.947791


In [90]:
import re

In [91]:
def preprocess_text_number(text):

  #Step 1:replace \d+ → NUMBER
  text = re.sub(
    r"\d+",
    "NUMBER",
    text
  )

  #Step 2:tokenize
  tokens = word_tokenize(text)

  #Step 3:lowercase
  lower_tokens = [
      token.lower()
      for token in tokens
  ]

  #Step 4:remove punctuation / non-alphabetic tokens
  clean_tokens = [
      token
      for token in lower_tokens
      if token.isalpha()
  ]

  #Step 5:remove stopwords
  filtered_tokens = [
      token
      for token in clean_tokens
      if token not in english_stopwords
  ]

  #Step 6: stemming
  stemmed_tokens = [
      stemmer.stem(token)
      for token in filtered_tokens
  ]

  processed_text = " ".join(stemmed_tokens)

  return processed_text

In [92]:
df["processed_message_number"] = df["message"].apply(
    preprocess_text_number
)

In [93]:
X_train_number = df.loc[
    X_train.index,
    "processed_message_number"
]

X_test_number = df.loc[
    X_test.index,
    "processed_message_number"
]

In [94]:
number_vectorizer = CountVectorizer()

In [95]:
X_train_number_bow = number_vectorizer.fit_transform(
    X_train_number
)

X_test_number_bow = number_vectorizer.transform(
    X_test_number
)

In [96]:
print(X_train_number_bow.shape)
print(X_test_number_bow.shape)

(4457, 5490)
(1115, 5490)


In [97]:
number_features = number_vectorizer.get_feature_names_out()

print(number_features[:50])

['aa' 'aah' 'aaniy' 'aathi' 'ab' 'abbey' 'abdomen' 'abeg' 'abel'
 'aberdeen' 'abi' 'abil' 'abiola' 'abj' 'abl' 'abnorm' 'abouta' 'absenc'
 'absolut' 'absolutli' 'abstract' 'abt' 'abta' 'aburo' 'abus' 'ac'
 'academ' 'acc' 'accent' 'accentur' 'accept' 'access' 'accid' 'accident'
 'accommodationvouch' 'accomod' 'accordin' 'accordingli' 'account'
 'accumul' 'ach' 'achan' 'acknowledg' 'aclnumberpm' 'acnt' 'aco' 'across'
 'act' 'actin' 'action']


In [98]:
number_model = LogisticRegression(
    max_iter=1000
)

In [99]:
number_model.fit(X_train_number_bow, y_train)

LogisticRegression(max_iter=1000)

In [100]:
y_pred_number = number_model.predict(X_test_number_bow)

print(y_pred_number[:20])

['ham' 'ham' 'ham' 'spam' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham'
 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham']


In [101]:
number_accuracy = accuracy_score(
    y_test,
    y_pred_number
)

number_precision = precision_score(
    y_test,
    y_pred_number,
    pos_label="spam"
)

number_recall = recall_score(
    y_test,
    y_pred_number,
    pos_label="spam"
)

number_f1 = f1_score(
    y_test,
    y_pred_number,
    pos_label="spam"
)

In [102]:
print(number_accuracy)
print(number_precision)
print(number_recall)
print(number_f1)

0.9838565022421525
0.9712230215827338
0.9060402684563759
0.9375


In [103]:
cm_bow = confusion_matrix(
    y_test,
    y_pred_number,
    labels=["ham", "spam"]
)

print(cm_bow)

[[962   4]
 [ 14 135]]


In [104]:
comparison_df = pd.DataFrame({
    "message": df.loc[X_test.index, "message"].reset_index(drop=True),
    "actual": y_test.reset_index(drop=True),
    "old_pred": y_pred_bow,
    "number_pred": y_pred_number
})

In [105]:
print(comparison_df)

                                                message actual old_pred  \
0       No need to buy lunch for me.. I eat maggi mee..    ham      ham   
1     Ok im not sure what time i finish tomorrow but...    ham      ham   
2     Waiting in e car 4 my mum lor. U leh? Reach ho...    ham      ham   
3     You have won ?1,000 cash or a ?2,000 prize! To...   spam     spam   
4           If you r @ home then come down within 5 min    ham      ham   
...                                                 ...    ...      ...   
1110  AH POOR BABY!HOPE URFEELING BETTERSN LUV! PROB...    ham      ham   
1111          O ic lol. Should play 9 doors sometime yo    ham      ham   
1112  Ambrith..madurai..met u in arun dha marrge..re...    ham      ham   
1113                    Dear umma she called me now :-)    ham      ham   
1114  Dont think so. It turns off like randomlly wit...    ham      ham   

     number_pred  
0            ham  
1            ham  
2            ham  
3           spam  
4   

In [106]:
rescued_spam = comparison_df[
    (comparison_df["actual"] == "spam") &
    (comparison_df["old_pred"] == "ham") &
    (comparison_df["number_pred"] == "spam")
]

In [107]:
for _, row in rescued_spam.iterrows():
    print("Message:", row["message"])
    print("Old:", row["old_pred"])
    print("NUMBER:", row["number_pred"])
    print("-----")

Message: it to 80488. Your 500 free text messages are valid until 31 December 2005.
Old: ham
NUMBER: spam
-----
Message: Can U get 2 phone NOW? I wanna chat 2 set up meet Call me NOW on 09096102316 U can cum here 2moro Luv JANE xx Calls£1/minmoremobsEMSPOBox45PO139WA
Old: ham
NUMBER: spam
-----
Message: 1000's of girls many local 2 u who r virgins 2 this & r ready 2 4fil ur every sexual need. Can u 4fil theirs? text CUTE to 69911(£1.50p. m)
Old: ham
NUMBER: spam
-----
Message: FreeMsg Hey U, i just got 1 of these video/pic fones, reply WILD to this txt & ill send U my pics, hurry up Im so bored at work xxx (18 150p/rcvd STOP2stop)
Old: ham
NUMBER: spam
-----


In [108]:
new_false_positive = comparison_df[
    (comparison_df["actual"] == "ham") &
    (comparison_df["old_pred"] == "ham") &
    (comparison_df["number_pred"] == "spam")
]

In [109]:
print(new_false_positive.shape)
print(new_false_positive["message"].iloc[0])

(4, 4)
Funny fact Nobody teaches volcanoes 2 erupt, tsunamis 2 arise, hurricanes 2 sway aroundn no 1 teaches hw 2 choose a wife Natural disasters just happens


## 10. Final Model Comparison


In [110]:
model_results = pd.DataFrame({
    "Model": [
        "BoW Unigram",
        "BoW + Bigrams",
        "TF-IDF Unigram",
        "TF-IDF + Bigrams",
        "BoW + NUMBER"
    ],
    "Accuracy": [
        bow_accuracy,
        ngram_accuracy,
        tfidf_accuracy,
        tfidf_ngram_accuracy,
        number_accuracy
    ],
    "Precision": [
        bow_precision,
        ngram_precision,
        tfidf_precision,
        tfidf_ngram_precision,
        number_precision
    ],
    "Recall": [
        bow_recall,
        ngram_recall,
        tfidf_recall,
        tfidf_ngram_recall,
        number_recall
    ],
    "F1": [
        bow_f1,
        ngram_f1,
        tfidf_f1,
        tfidf_ngram_f1,
        number_f1
    ]
})

model_results

,Model,Accuracy,Precision,Recall,F1
0,BoW Unigram,0.981166,0.977612,0.879195,0.925795
1,BoW + Bigrams,0.976682,0.992000,0.832215,0.905109
2,TF-IDF Unigram,0.961435,1.000000,0.711409,0.831373
3,TF-IDF + Bigrams,0.958744,0.990476,0.697987,0.818898
4,BoW + NUMBER,0.983857,0.971223,0.906040,0.937500


## 11. Conclusions

### Key Findings

Overall, the BoW + NUMBER model achieved the best overall performance, with an accuracy of 0.9839, recall of 0.9060, and F1-score of 0.9375. Although the TF-IDF unigram model achieved the highest precision, the BoW + NUMBER model provided the strongest overall balance of performance, particularly for detecting spam messages.

### Error Analysis and Feature Engineering

Error analysis revealed that many false negatives contained numeric information, such as phone numbers, short codes, and prices, that was removed during the original preprocessing pipeline. Further exploratory data analysis showed that only 15.65% of ham messages contained digits, compared with 94.78% of spam messages.

Based on this finding, digit sequences were normalized to a `NUMBER` token using regular expressions (regex). After incorporating this feature, spam recall increased from 0.8792 to 0.9060, while the number of false negatives decreased from 18 to 14. These results demonstrate how error analysis can guide feature engineering and improve classification performance.

### Limitations

The BoW + Logistic Regression approach relies primarily on surface-level lexical features and does not explicitly model word order, broader context, or semantics. As a result, messages containing unusual lexical or numeric patterns may still be misclassified.

### Future Improvements

Future work could distinguish different types of numeric information, such as `PHONE`, `MONEY`, `DATE`, and `NUMBER`, instead of mapping all digit sequences to a single token. Another direction would be to explore contextual embeddings or transformer-based classifiers that can better represent word order, context, and semantic information.